# vLLM

[vLLM](https://vllm.readthedocs.io/en/latest/index.html) 是一个快速易用的 LLM 推理和服务的库，提供：

* 行业领先的服务吞吐量
* 使用 PagedAttention 高效管理注意力键值内存
* 对传入请求进行连续批处理
* 优化的 CUDA 内核

本笔记本将介绍如何将 LLM 与 langchain 和 vLLM 结合使用。

要使用，您应该已安装 `vllm` Python 包。

In [1]:
%pip install --upgrade --quiet  vllm -q

In [1]:
from langchain_community.llms import VLLM

llm = VLLM(
    model="mosaicml/mpt-7b",
    trust_remote_code=True,  # mandatory for hf models
    max_new_tokens=128,
    top_k=10,
    top_p=0.95,
    temperature=0.8,
)

print(llm.invoke("What is the capital of France ?"))

INFO 08-06 11:37:33 llm_engine.py:70] Initializing an LLM engine with config: model='mosaicml/mpt-7b', tokenizer='mosaicml/mpt-7b', tokenizer_mode=auto, trust_remote_code=True, dtype=torch.bfloat16, use_dummy_weights=False, download_dir=None, use_np_weights=False, tensor_parallel_size=1, seed=0)
INFO 08-06 11:37:41 llm_engine.py:196] # GPU blocks: 861, # CPU blocks: 512


Processed prompts: 100%|██████████| 1/1 [00:00<00:00,  2.00it/s]


What is the capital of France ? The capital of France is Paris.


## 将模型集成到 LLMChain 中

LLMChain 是 LangChain 中用于将大型语言模型（LLM）与提示模板、输出解析器等组件结合使用的核心抽象。

以下是如何将模型集成到 LLMChain 中的步骤：

1.  **导入必要的库**

    ```python
    from langchain.llms import OpenAI
    from langchain.prompts import PromptTemplate
    from langchain.chains import LLMChain
    ```

2.  **初始化 LLM**

    首先，您需要初始化您要使用的 LLM。在本例中，我们将使用 OpenAI 的模型。确保您已设置 `OPENAI_API_KEY` 环境变量。

    ```python
    llm = OpenAI(temperature=0.9)
    ```

    *   `temperature` 参数控制生成文本的随机性。较高的值会产生更随机的输出，而较低的值会产生更确定性的输出。

3.  **创建提示模板**

    提示模板定义了您将如何格式化传递给 LLM 的输入。它允许您将变量插入到预定义的文本中。

    ```python
    prompt = PromptTemplate(
        input_variables=["product_name"],
        template="为 {product_name} 写一个朗朗上口的广告语。",
    )
    ```

    *   `input_variables` 是一个列表，其中包含提示模板中使用的变量名。
    *   `template` 是实际的提示文本，其中包含用花括号 `{}` 包围的变量。

4.  **创建 LLMChain**

    现在，您可以将 LLM 和提示模板组合到一个 LLMChain 中。

    ```python
    chain = LLMChain(llm=llm, prompt=prompt)
    ```

    *   `llm` 参数是您初始化的 LLM 实例。
    *   `prompt` 参数是您创建的提示模板实例。

5.  **运行 Chain**

    最后，您可以通过传递输入变量的值来运行 Chain。

    ```python
    product = "高效笔记本电脑"
    response = chain.run(product)
    print(response)
    ```

    这将生成一个广告语，例如：

    ```
    告别卡顿，拥抱流畅！这款高效笔记本电脑，让您的灵感永不停歇！
    ```

**更复杂的示例：**

您可以创建更复杂的提示模板，包含多个输入变量，并使用不同的 LLM 和输出解析器。

```python
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.output_parsers import SimpleJsonOutputParser
import json

# 1. 初始化 LLM
llm = OpenAI(temperature=0.7)

# 2. 创建提示模板
template = """
您是一位经验丰富的市场营销专家。请为以下产品构思三个不同的广告语，并以 JSON 格式返回，
其中包含 "slogan" 键，其值为广告语字符串。

产品名称：{product_name}
产品特点：{product_features}

JSON 输出：
"""
prompt = PromptTemplate(
    input_variables=["product_name", "product_features"],
    template=template,
)

# 3. 创建 LLMChain
# 注意：这里我们不直接使用输出解析器，因为我们希望 LLM 生成 JSON 字符串
chain = LLMChain(llm=llm, prompt=prompt)

# 4. 运行 Chain
product_name = "智能手表"
product_features = "长续航，健康监测，GPS定位"

# LLM 将返回一个 JSON 格式的字符串
response_str = chain.run({
    "product_name": product_name,
    "product_features": product_features
})

# 将 JSON 字符串解析为 Python 字典
try:
    response_json = json.loads(response_str)
    print(json.dumps(response_json, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("LLM 返回的不是有效的 JSON 格式:")
    print(response_str)

```

在此示例中：

*   我们使用了一个更详细的提示，要求 LLM 以 JSON 格式返回三个广告语。
*   `chain.run()` 方法现在接收一个字典，其中包含所有输入变量的值。
*   我们使用 `json.loads()` 将 LLM 返回的 JSON 字符串解析为 Python 字典，以便进一步处理。

LLMChain 是 LangChain 中一个非常灵活的组件，您可以根据您的具体需求组合不同的 LLM、提示模板和输出解析器来构建强大的应用程序。

In [3]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate.from_template(template)

llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "Who was the US president in the year the first Pokemon game was released?"

print(llm_chain.invoke(question))

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]



1. The first Pokemon game was released in 1996.
2. The president was Bill Clinton.
3. Clinton was president from 1993 to 2001.
4. The answer is Clinton.


## 分布式推理

vLLM 支持分布式张量并行推理和部署。

要使用 LLM 类运行多 GPU 推理，请将 `tensor_parallel_size` 参数设置为您想要使用的 GPU 数量。例如，要在 4 个 GPU 上运行推理

In [ ]:
from langchain_community.llms import VLLM

llm = VLLM(
    model="mosaicml/mpt-30b",
    tensor_parallel_size=4,
    trust_remote_code=True,  # mandatory for hf models
)

llm.invoke("What is the future of AI?")

## 量化

vLLM 支持 `awq` 量化。要启用它，请将 `quantization` 传递给 `vllm_kwargs`。

In [ ]:
llm_q = VLLM(
    model="TheBloke/Llama-2-7b-Chat-AWQ",
    trust_remote_code=True,
    max_new_tokens=512,
    vllm_kwargs={"quantization": "awq"},
)

## OpenAI 兼容服务器

vLLM 可以部署为一个模仿 OpenAI API 协议的服务器。这使得 vLLM 可以作为即插即用型替代品，用于使用 OpenAI API 的应用程序。

此服务器可以以与 OpenAI API 相同的格式进行查询。

### OpenAI 兼容的 Completion

In [3]:
from langchain_community.llms import VLLMOpenAI

llm = VLLMOpenAI(
    openai_api_key="EMPTY",
    openai_api_base="http://localhost:8000/v1",
    model_name="tiiuae/falcon-7b",
    model_kwargs={"stop": ["."]},
)
print(llm.invoke("Rome is"))

 a city that is filled with history, ancient buildings, and art around every corner


## LoRA 适配器
LoRA 适配器可与任何实现 `SupportsLoRA` 的 vLLM 模型一起使用。

In [ ]:
from langchain_community.llms import VLLM
from vllm.lora.request import LoRARequest

llm = VLLM(
    model="meta-llama/Llama-3.2-3B-Instruct",
    max_new_tokens=300,
    top_k=1,
    top_p=0.90,
    temperature=0.1,
    vllm_kwargs={
        "gpu_memory_utilization": 0.5,
        "enable_lora": True,
        "max_model_len": 350,
    },
)
LoRA_ADAPTER_PATH = "path/to/adapter"
lora_adapter = LoRARequest("lora_adapter", 1, LoRA_ADAPTER_PATH)

print(
    llm.invoke("What are some popular Korean street foods?", lora_request=lora_adapter)
)